In [0]:
bronze_path   = 'abfss://bikestore@azurelakehouse.dfs.core.windows.net/bronze/'
silver_path   = 'abfss://bikestore@azurelakehouse.dfs.core.windows.net/silver/'
gold_path     = 'abfss://bikestore@azurelakehouse.dfs.core.windows.net/gold/'
resource_path = 'abfss://bikestore@azurelakehouse.dfs.core.windows.net/resource/'

In [0]:

# criar várias tabelas temporárias de forma prática
bronze_map = {
    "tmp_bronze_brands":     f"{bronze_path}/brands/",
    "tmp_bronze_categories": f"{bronze_path}/categories/",
    "tmp_bronze_customers":  f"{bronze_path}/customers/",
    "tmp_bronze_order_items":f"{bronze_path}/order_items/",
    "tmp_bronze_orders":     f"{bronze_path}/orders/",
    "tmp_bronze_products":   f"{bronze_path}/products/",
    "tmp_bronze_staffs":     f"{bronze_path}/staffs/",
    "tmp_bronze_stocks":     f"{bronze_path}/stocks/",
    "tmp_bronze_stores":     f"{bronze_path}/stores/",
}

for view_name, path in bronze_map.items():
    (spark.read.format('delta')
        .load(path)
        .createOrReplaceTempView(view_name)
    )


In [0]:
df_orders_silver = spark.sql("""
with order_items as (
  select
    order_id,
    item_id,
    product_id,
    quantity,
    list_price,
    discount,
    cast(list_price * quantity * discount as decimal(10, 2)) as discount_amount,
    cast(list_price * quantity * (1 - discount) as decimal(10, 2)) as total_sale
  from
    tmp_bronze_order_items
)
select
  ord.order_id,
  ord.customer_id,
  case
    when ord.order_status = 1 then 'Pending'
    when ord.order_status = 2 then 'Processing'
    when ord.order_status = 3 then 'Shipped'
    when ord.order_status = 4 then 'Delivered'
    else 'unknown'
  end as status,
  ord.order_status,
  ord.order_date,
  ord.required_date,
  ord.shipped_date,
  ord.store_id,
  ord.staff_id,
  st.store_name,
  st.city,
  st.state,
  stf.first_name as staff_name,
  stf.email as staff_email,
  stf.active as staff_active,
  product_id,
  quantity,
  list_price,
  discount,
  discount_amount,
  total_sale
from
  tmp_bronze_orders as ord
    left join tmp_bronze_stores as st
      on ord.store_id = st.store_id
    left join tmp_bronze_staffs as stf
      on ord.staff_id = stf.staff_id
    left join order_items as oi
      on ord.order_id = oi.order_id

""" )


#salvar em parquet como delta tabela de teste SILVER
df_orders_silver.write\
    .mode('overwrite')\
    .format('delta')\
    .option('mergeSchema', 'true')\
    .save(f'{silver_path}/orders')